This notebook shows the 66.66% reduction in latency on the credit card fraud dataset from https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud. The method is early-exiting and building a cascaded system of models and this notebook demonstrates that reductions in compute-time are possible while preserving accuracy. 

In [1]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from sklearn.neural_network import MLPClassifier
from utils import analyze, optimize, HyperplaneDecisionRealTime, Router, PathConfig, RouterConfig
from sklearn.model_selection import train_test_split
import time
import os
import numpy as np
from tqdm import tqdm

import warnings

In [3]:


# Suppress only RuntimeWarning
warnings.filterwarnings("ignore", category=RuntimeWarning)

def load_dataset():
    df = pd.read_csv('/Users/samrandall/Downloads/creditcard.csv')
    X = df[[col for col in df.columns if col.startswith('V')]].to_numpy()
    y = df['Class'].to_numpy()
    return X, y




Load the dataset and train a simple MLPClassifier. This will take 10 seconds.

In [4]:
X, y = load_dataset()
X_train, X_test, y_train, y_test = train_test_split(X, y)
model = MLPClassifier()
model.fit(X_train, y_train)

MLPClassifier()

In [11]:
json_save_path = "creditcard_hyperplane.json"


# Load the rule from cached.
if os.path.exists(json_save_path):

    positive_path = PathConfig(path_type="class", activation=0)
    negative_path = PathConfig(path_type="default", activation=lambda x: model.predict(x).item())
    router_config = RouterConfig(positive_path=positive_path, negative_path=negative_path)
    hd = HyperplaneDecisionRealTime.load(json_save_path)
    router = Router(hd, router_config.positive_path, router_config.negative_path)
else:
    router = analyze(model, X_train)
    router.decision.save(json_save_path)

## The rule that the moco anaylsis system found. 

In [12]:
print("Rule information: W: {}, b: {}, t: {}".format(router.decision.W, router.decision.b, router.decision.t))

Rule information: W: [[ 0.15805711]
 [-0.03494076]
 [ 0.047534  ]
 [-0.45371963]
 [-0.09405791]
 [ 0.05795528]
 [-0.03107688]
 [ 0.18404747]
 [ 0.26969535]
 [ 0.5797903 ]
 [-0.00520772]
 [-0.06015844]
 [ 0.3161317 ]
 [ 0.7006523 ]
 [ 0.31168193]
 [ 0.40475035]
 [-0.21704558]
 [ 0.0986935 ]
 [-0.32770777]
 [ 0.13570174]
 [-0.35769273]
 [-0.65175182]
 [ 0.08622373]
 [-0.00886996]
 [ 0.34782567]
 [-0.28369063]
 [ 0.38647513]
 [ 0.08411804]], b: [9.16485614], t: 8.524070562469952


In [13]:

for i in range(len(X)):
    router.forward(X[i:i+1])

start = time.time()
for i in tqdm(range(len(X))):
    router.forward(X[i:i+1])
end = time.time()
early_exit_time = end - start

for i in range(len(X)):
    model.predict(X[i:i+1])

start = time.time()
for i in tqdm(range(len(X))):
    model.predict(X[i:i+1])
end = time.time()
original_time = end - start

print("Original Time", original_time) # 13.354 seconds
print("Early Exit Time", early_exit_time) # 10.707 seconds

100%|██████████| 284807/284807 [00:12<00:00, 22618.07it/s]

Original Time 12.592881917953491
Early Exit Time 4.513746976852417


Result: Worst Case Latency is unchanged. In this case, it actually improves.

In [14]:
start = time.time()

for i in range(len(X)):
    router.forward(X[i:i+1])
worst_case_latency = 0
routed_latencies = []
for i in tqdm(range(len(X))):
    
    s = time.time()
    router.forward(X[i:i+1])
    e = time.time()
    worst_case_latency = max(worst_case_latency, e - s)
    routed_latencies.append(e - s)


print("Routed", worst_case_latency)

for i in range(len(X)):
    model.predict(X[i:i+1])

base_latencies = []
worst_case_latency = 0
for i in tqdm(range(len(X))):
    s = time.time()
    model.predict(X[i:i+1])
    e = time.time()
    base_latencies.append(e - s)
    worst_case_latency = max(worst_case_latency, e - s)

print("original", worst_case_latency)

100%|██████████| 284807/284807 [00:04<00:00, 61440.97it/s]


Routed 0.0009891986846923828


100%|██████████| 284807/284807 [00:12<00:00, 22059.15it/s]

original 0.0018520355224609375


Counting how many data points avoided full model execution (were exited early)

In [17]:
decisions = []
for item in X:
    decisions.append(router.decision.forward(item[np.newaxis, :]))
decisions = np.array(decisions)

In [18]:
decisions.mean()

np.float64(0.7104846439869806)

## Latency 
The results in this run are that the original model took 14.04 seconds to run through the entire dataset of 284807 seconds. Compare this to Early Exit Latency of 4.68 seconds. This is a 66.7% reduction in CPU-time. Results vary based on the run of the algorithm. This is because 71% of the data points avoided model execution.

## Accuracy

This section shows below that the predictions in the routed model are 100% across the test set.

In [19]:
p = []
for item in X_test:
    p.append(router.forward(item[np.newaxis, :]))

early_exit_predictions = np.array(p)
original_predictions = model.predict(X_test)

In [20]:
np.mean(early_exit_predictions == original_predictions)

np.float64(1.0)